# Voxelmap-Pointshell Usage Examples

This notebook shows the basic workflow of the Python reimplementation:

- load a mesh
- generate a voxelmap
- generate a pointshell
- visualize the structures
- run proximity queries
- run collision queries

The examples use the sample model in `data/models/monkey.stl`.

In [ ]:
from pathlib import Path
import sys

import numpy as np

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from python.mesh import Mesh
from python.generate_voxelmap import generate_voxelmap
from python.generate_pointshell import generate_pointshell
from python.proximity_query import proximity_query
from python.collision_detection import detect_collision
from python.viewer import (
    create_mesh_scene,
    create_pointshell_scene,
    create_query_scene,
    create_voxelmap_scene,
)


## 1. Load a mesh

In [ ]:
mesh_path = PROJECT_ROOT / "data" / "models" / "monkey.stl"
mesh = Mesh.from_file(mesh_path)

print(f"source: {mesh.source_path}")
print(f"vertices: {mesh.vertex_count}")
print(f"triangles: {mesh.triangle_count}")
print(f"bounds:\n{mesh.bounds}")
print(f"centroid: {mesh.centroid}")


## 2. Generate a voxelmap

In [ ]:
voxel_size = 0.2
voxelmap = generate_voxelmap(mesh, voxel_size=voxel_size)

print(f"shape: {voxelmap.shape}")
print(f"voxel size: {voxelmap.voxel_size}")
print(f"origin: {voxelmap.origin}")
print(f"surface voxels: {(voxelmap.values == 0).sum()}")
print(f"interior voxels: {(voxelmap.values > 0).sum()}")
print(f"exterior voxels: {(voxelmap.values < 0).sum()}")


## 3. Generate a pointshell

In [ ]:
pointshell = generate_pointshell(mesh, voxel_size=voxel_size, target_spheres=32)

print(f"points: {pointshell.point_count}")
print(f"spheres: {pointshell.sphere_count}")
print(f"pointshell centroid: {pointshell.centroid}")
print(f"first sphere: {pointshell.sphere(0)}")


## 4. Visualization examples

The viewer helpers return `trimesh.Scene` objects. In a local desktop session you can usually inspect them with `.show()`.

If your environment does not support interactive windows, the scene objects are still useful for debugging and exporting.

In [ ]:
mesh_scene = create_mesh_scene(mesh)
voxel_scene = create_voxelmap_scene(voxelmap, mode="surface")
pointshell_scene = create_pointshell_scene(pointshell)

print(mesh_scene)
print(voxel_scene)
print(pointshell_scene)

# Uncomment these in an interactive environment:
# mesh_scene.show()
# voxel_scene.show()
# pointshell_scene.show()


## 5. Proximity query example

We move the pointshell slightly relative to the voxelmap and ask for the top closest/deepest points.

In [ ]:
transform = np.eye(4)
transform[:3, 3] = np.array([0.1, 0.0, 0.0])

query = proximity_query(
    voxelmap,
    pointshell,
    transform=transform,
    n=5,
    sphere_percentage=0.5,
    point_percentage=0.5,
)

print("sphere traversal order:", query.sphere_order[:10])
print("top hits:")
for hit in query.hits:
    print(
        {
            "point_id": hit.point_id,
            "sphere_id": hit.sphere_id,
            "signed_distance": hit.signed_distance,
            "position": hit.position.round(4),
            "normal": hit.normal.round(4),
        }
    )


## 6. Collision query example

The collision query reuses the proximity stage and accumulates a penalty-based force and torque over colliding points only.

In [ ]:
collision = detect_collision(
    voxelmap,
    pointshell,
    transform=transform,
    stiffness=2.0,
    sphere_percentage=0.5,
    point_percentage=0.5,
)

print(f"is colliding: {collision.is_colliding}")
print(f"colliding point count: {collision.colliding_point_ids.size}")
print(f"penetration stats: {collision.penetrations[:10]}")
print(f"total force: {collision.total_force}")
print(f"total torque: {collision.total_torque}")


## 7. Visualize query results

In [ ]:
query_scene = create_query_scene(
    voxelmap,
    pointshell,
    query,
    transform=transform,
)

print(query_scene)

# Uncomment in an interactive environment:
# query_scene.show()


## 8. Serialization examples

In [ ]:
tmp_dir = PROJECT_ROOT / "notebooks" / "_generated_examples"
tmp_dir.mkdir(exist_ok=True)

voxelmap_path = tmp_dir / "monkey_voxelmap.json"
pointshell_path = tmp_dir / "monkey_pointshell.json"

voxelmap.to_ascii(voxelmap_path)
pointshell.to_ascii(pointshell_path)

print(voxelmap_path)
print(pointshell_path)
